In [0]:
dbutils.widgets.text("catalog_name", "", "Catalog Name")
dbutils.widgets.dropdown("function", "upgrade", ["upgrade", "downgrade"], "Function")

In [0]:
catalog=dbutils.widgets.get("catalog_name")
function=dbutils.widgets.get("function")

In [0]:
"""
Migration: Create test_migration table
Revision: 001_create_test_migration
"""

def upgrade():
    """
    Create test_migration table in silver schema
    """
    print(f"Applying migration: Create test_migration table")
    print("-" * 80)
    
    # Create silver schema if it doesn't exist
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS {catalog}.silver
        COMMENT 'Silver layer - cleaned and validated data'
    """)
    print(f"✓ Schema ready: {catalog}.silver")
    
    # Create test_migration table
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.silver.test_migration (
            id STRING NOT NULL,
            name STRING,
            value INT
        )
        USING DELTA
        COMMENT 'Test migration table'
        TBLPROPERTIES (
            'delta.enableChangeDataFeed' = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
    """)
    print(f"✓ Created table: {catalog}.silver.test_migration")
    
    # Insert sample data
    spark.sql(f"""
        INSERT INTO {catalog}.silver.test_migration (id, name, value)
        VALUES 
            ('TEST001', 'Sample 1', 100),
            ('TEST002', 'Sample 2', 200),
            ('TEST003', 'Sample 3', 300)
    """)
    print(f"✓ Inserted 3 sample records")
    
    print("-" * 80)
    print(f"✓ Migration applied successfully")


def downgrade():
    """
    Drop test_migration table
    """
    print(f"Reverting migration: Create test_migration table")
    print("-" * 80)
    
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.silver.test_migration")
    print(f"✓ Dropped table: {catalog}.silver.test_migration")
    
    print("-" * 80)
    print(f"✓ Migration reverted successfully")

In [0]:
if function == "downgrade":
    downgrade()
else:
    upgrade()